In [ ]:
# Install required packages
!pip install numpy scikit-learn torch -q
!pip install rdkit xgboost -q
!pip install pandas

!pip install torch
!pip install transformers
!pip install sentencepiece

!pip install chemprop -q

In [ ]:
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

In [ ]:
import pandas as pd
import numpy as np
train_data = pd.read_csv('train.csv')


In [ ]:
#Import models
import numpy as np
import torch
from transformers import T5Tokenizer, T5EncoderModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

tokenizer = T5Tokenizer.from_pretrained('Rostlab/prot_t5_xl_half_uniref50-enc')
model = T5EncoderModel.from_pretrained(
    "Rostlab/prot_t5_xl_half_uniref50-enc",
    torch_dtype=torch.float16
)
model = model.to(device).eval()


In [ ]:
#Compute the embedding given a certain sequence
protein_cache = {}
def get_protein_embedding(sequences, batch_size = 2):
  uncached = [seq for seq in sequences if seq not in protein_cache]

  for start in range(0, len(uncached), batch_size):
    batch = uncached[start:start + batch_size]

    batch_spaced = [" ".join(list(seq)) for seq in batch]

    inputs = tokenizer(batch_spaced, return_tensors="pt", padding = True, truncation = True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    residue_embeddings = outputs.last_hidden_state
    attention_mask = inputs["attention_mask"].unsqueeze(-1)

    masked_embeddings = residue_embeddings * attention_mask
    sum_embeddings = masked_embeddings.sum(dim=1)

    batch_embeddings = (sum_embeddings / attention_mask.sum(dim=1)).cpu().numpy()

    for seq, emb in zip(batch, batch_embeddings):
      protein_cache[seq] = emb


In [ ]:
df       = train_data.copy()
protein  = df["amino_acid_sequence"].values
drugs    = df["SMILES"].values
affinity = df["Affinity"].values
valid_protein = [seq for seq in protein if isinstance(seq, str) and len(seq) > 0]
unique_proteins = list(set(valid_protein))

valid_mask = np.array([isinstance(seq, str) and len(seq) > 0 for seq in protein])
df         = df[valid_mask].reset_index(drop=True)
protein    = df["amino_acid_sequence"].values
drugs      = df["SMILES"].values
affinity   = df["Affinity"].values

unique_proteins = list(set(protein))
print(f"Unique proteins: {len(unique_proteins)}, Total rows: {len(protein)}")

get_protein_embedding(unique_proteins, batch_size=16)

seq_to_emb = {seq: protein_cache[seq] for seq in unique_proteins}

train_data["protein_emb"] = train_data["amino_acid_sequence"].map(seq_to_emb)

np.save("protein_embeddings.npy",
        np.array(list(seq_to_emb.values())).astype(np.float32))
print(f"Unique proteins embedded: {len(seq_to_emb)}")

In [ ]:
del model
torch.cuda.empty_cache()
import gc
gc.collect()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

In [ ]:
# Save embeddings to skip future computation
from google.colab import drive
drive.mount('/content/drive')

import pickle
with open("/content/drive/MyDrive/seq_to_emb.pkl", "wb") as f:
    pickle.dump(seq_to_emb, f)

train_data.to_pickle("/content/drive/MyDrive/train_with_embeddings.pkl")
print(f"Saved {len(seq_to_emb)} protein embeddings to Drive!")

In [ ]:
train_data.head()

In [ ]:
import pickle
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

# load everything from drive
train_data = pd.read_csv('train.csv')
train_data = train_data.dropna(subset=["amino_acid_sequence"]).reset_index(drop=True)

with open("/content/drive/MyDrive/seq_to_emb.pkl", "rb") as f:
    seq_to_emb = pickle.load(f)

# filter to only rows with known protein embeddings
valid_mask = train_data["amino_acid_sequence"].isin(seq_to_emb.keys())
train_data = train_data[valid_mask].reset_index(drop=True)
print(f"Rows with embeddings: {len(train_data)}")

In [ ]:
from chemprop.data import MoleculeDatapoint, MoleculeDataset
from rdkit import Chem

# filter invalid SMILES
valid_mask  = train_data["SMILES"].apply(lambda s: Chem.MolFromSmiles(s) is not None)
train_data  = train_data[valid_mask].reset_index(drop=True)
print(f"Rows after filtering: {len(train_data)}")

datapoints  = [MoleculeDatapoint.from_smi(s) for s in train_data["SMILES"]]
dataset     = MoleculeDataset(datapoints)

X_proteins  = np.array([seq_to_emb[seq] for seq in train_data["amino_acid_sequence"]]).astype(np.float32)
targets_np  = np.array(train_data["Affinity"]).astype(np.float32)

print(f"Rows: {len(dataset)}, Proteins: {X_proteins.shape}, Targets: {targets_np.shape}")

In [ ]:
import numpy as np
import pandas as pd
import torch
from chemprop.models import MPNN
from chemprop.nn import BondMessagePassing, MeanAggregation
from chemprop.data import build_dataloader, MoleculeDataset
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# --- get mol embeddings from chemprop ---
mp  = BondMessagePassing().to(device)
agg = MeanAggregation().to(device)
mp.eval()
agg.eval()

mol_embeddings = []
loader = build_dataloader(dataset, batch_size=64, shuffle=False)
with torch.no_grad():
    for batch in loader:
        bmg = batch.bmg
        bmg.V          = bmg.V.to(device)
        bmg.E          = bmg.E.to(device)
        bmg.edge_index = bmg.edge_index.to(device)
        bmg.batch      = bmg.batch.to(device)  # this was missing
        h   = mp(bmg)
        emb = agg(h, bmg.batch)
        mol_embeddings.append(emb.cpu().numpy())

X_mols = np.concatenate(mol_embeddings)
print(f"Mol embeddings shape: {X_mols.shape}")

# --- combine features same as before ---
def combine_features(X_mols, X_proteins):
    X = []
    for d, p in zip(X_mols, X_proteins):
        d = np.asarray(d)
        p = np.asarray(p)
        # no interaction term since dims don't match (300 vs 1024)
        x = np.concatenate([d, p])
        X.append(x)
    return np.stack(X)

X_combined = combine_features(X_mols, X_proteins)
y = targets_np

# --- split 70/15/15 ---
proteins    = train_data["amino_acid_sequence"].values
unique_prots = list(set(proteins))

train_prot, temp_prot = train_test_split(unique_prots, test_size=0.30, random_state=42)
val_prot,   test_prot = train_test_split(temp_prot,    test_size=0.50, random_state=42)

train_mask = np.array([p in set(train_prot) for p in proteins])
val_mask   = np.array([p in set(val_prot)   for p in proteins])
test_mask  = np.array([p in set(test_prot)  for p in proteins])

X_reg_train, y_reg_train = X_combined[train_mask], y[train_mask]
X_reg_valid, y_reg_valid = X_combined[val_mask],   y[val_mask]
X_reg_test,  y_reg_test  = X_combined[test_mask],  y[test_mask]


# --- models ---
reg_models = {
    "XGBoost": XGBRegressor(
    n_estimators=5000,
    max_depth=6,
    learning_rate=0.02,
    subsample=0.8,
    objective="reg:squarederror",
    colsample_bytree=0.7,
    tree_method="hist",
    early_stopping_rounds=50,
    reg_lambda=1.0,
    random_state=42
),
}



In [ ]:
# --- train ---
reg_results = {}
for name, model in reg_models.items():
    print(f"Training {name}...")
    if name == "XGBoost":
        model.fit(X_reg_train, y_reg_train,
                  eval_set=[(X_reg_valid, y_reg_valid)],
                  verbose=100)
    else:
        model.fit(X_reg_train, y_reg_train)
    reg_results[name] = {
        "model":       model,
        "train_preds": model.predict(X_reg_train),
        "valid_preds": model.predict(X_reg_valid),
        "test_preds":  model.predict(X_reg_test),
    }
print("Done.")


In [ ]:
def evaluate_regression(y_true, y_pred):
    return {
        'MSE': mean_squared_error(y_true, y_pred),
        'MAE': mean_absolute_error(y_true, y_pred),
        'R2':  r2_score(y_true, y_pred),
    }
# --- sklearn results ---
print(f"\n{'Model':<22} {'Split':<8} {'MSE':>8} {'MAE':>8} {'R2':>8}")
print("-" * 58)
for name, res in reg_results.items():
    metrics = evaluate_regression(y_reg_valid, res['valid_preds'])
    print(f"{name:<22} {'Valid':<8} {metrics['MSE']:>8.2f} {metrics['MAE']:>8.2f} {metrics['R2']:>8.3f}")

In [ ]:
# import torch
# import torch.nn as nn
# from torch.utils.data import TensorDataset, DataLoader
# from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# def evaluate_regression(y_true, y_pred):
#     return {
#         'MSE': mean_squared_error(y_true, y_pred),
#         'MAE': mean_absolute_error(y_true, y_pred),
#         'R2':  r2_score(y_true, y_pred),
#     }

# # --- tensors ---
# X_train_t = torch.tensor(X_reg_train, dtype=torch.float32).to(device)
# X_valid_t = torch.tensor(X_reg_valid, dtype=torch.float32).to(device)
# y_train_t = torch.tensor(y_reg_train, dtype=torch.float32).to(device)

# # --- dataloaders ---
# train_ds     = TensorDataset(X_train_t, y_train_t)
# train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)

# # --- MLP ---
# class MLP(nn.Module):
#     def __init__(self, input_dim):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(input_dim, 512),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(512, 256),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(256, 1)
#         )
#     def forward(self, x):
#         return self.net(x).squeeze(1)

# mlp       = MLP(input_dim=X_train_t.shape[1]).to(device)
# optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3)
# loss_fn   = nn.MSELoss()
# best_r2   = -float('inf')

# for epoch in range(100):
#     mlp.train()
#     for X_batch, y_batch in train_loader:
#         optimizer.zero_grad()
#         loss = loss_fn(mlp(X_batch), y_batch)
#         loss.backward()
#         optimizer.step()

#     if (epoch + 1) % 10 == 0:
#         mlp.eval()
#         with torch.no_grad():
#             val_preds = mlp(X_valid_t).cpu().numpy()
#         metrics = evaluate_regression(y_reg_valid, val_preds)
#         if metrics['R2'] > best_r2:
#             best_r2 = metrics['R2']
#             torch.save(mlp.state_dict(), "best_mlp.pt")
#         print(f"Epoch {epoch+1:3d} | R²: {metrics['R2']:.4f} | MSE: {metrics['MSE']:.4f} | MAE: {metrics['MAE']:.4f}")

# print(f"\nBest MLP R²: {best_r2:.4f}")

In [ ]:
# from chemprop.data import build_dataloader, MoleculeDataset

# # split indices
# n         = len(targets_np)
# indices   = np.random.permutation(n)
# split     = int(0.9 * n)
# train_idx = indices[:split]
# val_idx   = indices[split:]

# # rebuild datasets from scratch using the indices
# train_smiles = [train_data["SMILES"].iloc[i] for i in train_idx]
# val_smiles   = [train_data["SMILES"].iloc[i] for i in val_idx]

# dataset_train = MoleculeDataset([MoleculeDatapoint.from_smi(s) for s in train_smiles])
# dataset_val   = MoleculeDataset([MoleculeDatapoint.from_smi(s) for s in val_smiles])

# X_prot_train  = torch.tensor(X_proteins[train_idx], dtype=torch.float32).to(device)
# X_prot_val    = torch.tensor(X_proteins[val_idx],   dtype=torch.float32).to(device)
# y_train       = torch.tensor(targets_np[train_idx], dtype=torch.float32).to(device)
# y_val         = torch.tensor(targets_np[val_idx],   dtype=torch.float32).to(device)

# train_loader = build_dataloader(dataset_train, batch_size=64, shuffle=False)
# val_loader   = build_dataloader(dataset_val,   batch_size=64, shuffle=False)

# # model: trainable GNN + MLP predictor
# class DTIModel(nn.Module):
#     def __init__(self, mol_dim, prot_dim):
#         super().__init__()
#         self.mp  = BondMessagePassing()
#         self.agg = MeanAggregation()
#         self.mlp = nn.Sequential(
#             nn.Linear(mol_dim + prot_dim, 512),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(512, 256),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(256, 1)
#         )

#     def forward(self, bmg, prot_emb):
#         bmg.V          = bmg.V.to(device)
#         bmg.E          = bmg.E.to(device)
#         bmg.edge_index = bmg.edge_index.to(device)
#         bmg.batch      = bmg.batch.to(device)
#         h   = self.mp(bmg)
#         mol = self.agg(h, bmg.batch)
#         x   = torch.cat([mol, prot_emb], dim=1)
#         return self.mlp(x).squeeze(1)

# dti_model = DTIModel(mol_dim=300, prot_dim=1024).to(device)
# optimizer  = torch.optim.Adam(dti_model.parameters(), lr=1e-3)
# loss_fn    = nn.MSELoss()
# best_r2    = -float('inf')

# for epoch in range(50):
#     dti_model.train()
#     for i, batch in enumerate(train_loader):
#         bmg      = batch.bmg
#         prot_emb = X_prot_train[i*64:(i+1)*64]
#         y_batch  = y_train[i*64:(i+1)*64]
#         optimizer.zero_grad()
#         pred = dti_model(bmg, prot_emb)
#         loss = loss_fn(pred, y_batch)
#         loss.backward()
#         optimizer.step()

#     dti_model.eval()
#     preds_all, targets_all = [], []
#     with torch.no_grad():
#         for i, batch in enumerate(val_loader):
#             bmg      = batch.bmg
#             prot_emb = X_prot_val[i*64:(i+1)*64]
#             pred     = dti_model(bmg, prot_emb).cpu().numpy()
#             preds_all.append(pred)
#             targets_all.append(targets_np[val_idx][i*64:(i+1)*64])

#     preds_all   = np.concatenate(preds_all)
#     targets_all = np.concatenate(targets_all)
#     metrics     = evaluate_regression(targets_all, preds_all)

#     if metrics['R2'] > best_r2:
#         best_r2 = metrics['R2']
#         torch.save(dti_model.state_dict(), "best_dti.pt")

#     if (epoch + 1) % 10 == 0:
#         print(f"Epoch {epoch+1:3d} | R²: {metrics['R2']:.4f} | MSE: {metrics['MSE']:.4f} | MAE: {metrics['MAE']:.4f}")

# print(f"\nBest R²: {best_r2:.4f}")

In [ ]:
import pickle
from google.colab import drive
drive.mount('/content/drive')

# save XGBoost from this notebook with a distinct name
with open("/content/drive/MyDrive/xgb_fg_fragments_331k.pkl", "wb") as f:
    pickle.dump(reg_results["XGBoost"]["model"], f)

print("Saved xgb_fg_fragments_331k.pkl!")

In [ ]:
import pickle
with open("/content/drive/MyDrive/frag_cache.pkl", "rb") as f:
    frag_cache = pickle.load(f)
print(f"Loaded {len(frag_cache)} cached fragments")